# Contact-Predicted SAAP Analysis

Tests whether contact-predicted SAAP peptides (proximal to destabilizing missense mutations) are:
1. **Detected more often** in TMT channels whose patient carries the corresponding destabilizing missense
2. **More abundant** (higher RI intensity) in carrier channels vs non-carrier channels

Compares destabilizing (AM+SPURS, SPURS only, AM only) vs neutral contact predictions as a negative control.

**Logic:**
- Contact FASTA entries tagged `destab_contact` or `neutral_contact` in the description
- For each detected contact PSM in a plex:
  - Parse gene + contact position from the entry name
  - Find which plex patients have a destabilizing missense within 3Å of that position
  - Compare RI intensity: carrier channels vs non-carrier channels
  - Compute detection rate per channel group

In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# ── CONFIG ────────────────────────────────────────────────────────────────────
REPO_DIR     = '/home/leduc.an/AAS_Evo_project/AAS_Evo'
TMT_MAP      = f'{REPO_DIR}/metadata/PDC_meta/pdc_file_tmt_map.tsv'
GDC_META     = f'{REPO_DIR}/metadata/GDC_meta/gdc_meta_matched.tsv'
MISSENSE     = '/scratch/leduc.an/AAS_Evo/ANALYSIS/all_missense_with_spurs.tsv'
REF_FASTA    = '/scratch/leduc.an/AAS_Evo/SEQ_FILES/uniprot_human_canonical.fasta'
RESULTS_BASE = '/scratch/leduc.an/AAS_Evo/MS_SEARCH/results_contact'
CONTACT_DIR  = '/scratch/leduc.an/AAS_Evo/SPURS/contact_maps'
DDG_DIR      = '/scratch/leduc.an/AAS_Evo/SPURS/ddg_matrices'
PLEX_LIST    = '/scratch/leduc.an/AAS_Evo/MS_SEARCH/plex_list.txt'
OUT_DIR      = '/scratch/leduc.an/AAS_Evo/ANALYSIS/contact_saap'
os.makedirs(OUT_DIR, exist_ok=True)

AM_THRESHOLD    = 0.564
SPURS_THRESHOLD = 0.5
AM_BENIGN_MAX   = 0.1
GNOMAD_NEUTRAL  = 0.1
GNOMAD_MAX      = 0.01
VAF_THRESHOLD   = 0.3
DIST_THRESHOLD  = 3.0   # Å Cα-Cα (must match generate_contact_saap_fastas.py)

TMT_CHANNEL_MAP = {
    'tmt_126':'126','tmt_127n':'127N','tmt_127c':'127C',
    'tmt_128n':'128N','tmt_128c':'128C','tmt_129n':'129N',
    'tmt_129c':'129C','tmt_130n':'130N','tmt_130c':'130C',
    'tmt_131':'131N','tmt_131c':'131C','tmt_126c':'126C','tmt_134n':'134N',
}
CHANNEL_ORDER = ['126','127N','127C','128N','128C','129N','129C','130N','130C','131N','131C']


In [ ]:
# ── LOAD METADATA ─────────────────────────────────────────────────────────────
tmt = pd.read_csv(TMT_MAP, sep='\t')
gdc = pd.read_csv(GDC_META, sep='\t')
if 'file_id' in gdc.columns and 'gdc_file_id' not in gdc.columns:
    gdc = gdc.rename(columns={'file_id':'gdc_file_id'})

with open(PLEX_LIST) as f:
    plex_ids = [l.strip() for l in f if l.strip()]
# Only plexes with results_contact results
plex_ids = [p for p in plex_ids
            if os.path.exists(os.path.join(RESULTS_BASE, p, 'combined_protein.tsv'))]

uuid_to_stype = gdc.set_index('gdc_file_id')['sample_type'].to_dict()
case_sample_to_uuid = gdc.set_index(['case_submitter_id','sample_type'])['gdc_file_id'].to_dict()

print(f'Plexes with results_contact: {len(plex_ids)}')

In [ ]:
# ── LOAD & FILTER MISSENSE TABLE ──────────────────────────────────────────────
print('Loading missense table...')
miss = pd.read_csv(MISSENSE, sep='\t', low_memory=False,
                   usecols=['sample_id','SYMBOL','Protein_position',
                            'Amino_acids','VAF','gnomADe_AF',
                            'am_pathogenicity','spurs_ddg'])
miss['VAF']             = pd.to_numeric(miss['VAF'],             errors='coerce')
miss['gnomADe_AF']      = pd.to_numeric(miss['gnomADe_AF'],      errors='coerce').fillna(0)
miss['am_pathogenicity']= pd.to_numeric(miss['am_pathogenicity'],errors='coerce')
miss['spurs_ddg']       = pd.to_numeric(miss['spurs_ddg'],       errors='coerce')
miss['pos']             = pd.to_numeric(
    miss['Protein_position'].astype(str).str.split('-').str[0], errors='coerce')

base_ok   = (miss['VAF'] >= VAF_THRESHOLD) & (miss['gnomADe_AF'] < GNOMAD_MAX)
am_pos    = miss['am_pathogenicity'] >= AM_THRESHOLD
spurs_pos = miss['spurs_ddg']        >= SPURS_THRESHOLD

def classify_miss(am, spurs):
    a = pd.notna(am)    and am    >= AM_THRESHOLD
    s = pd.notna(spurs) and spurs >= SPURS_THRESHOLD
    if a and s:  return 'Both AM+SPURS'
    if s:        return 'SPURS only'
    if a:        return 'AM only'
    if pd.notna(am) and am <= AM_BENIGN_MAX: return 'Neutral'
    return None

destab  = miss[(am_pos | spurs_pos) & base_ok].copy()
neutral = miss[(miss['am_pathogenicity'] <= AM_BENIGN_MAX) &
               (miss['gnomADe_AF'] >= GNOMAD_NEUTRAL)].copy()

destab['group']  = destab.apply(lambda r: classify_miss(r['am_pathogenicity'], r['spurs_ddg']), axis=1)
neutral['group'] = 'Neutral'
all_miss = pd.concat([destab, neutral], ignore_index=True)
all_miss_indexed = all_miss.groupby('sample_id')

all_processed_uuids = set(miss['sample_id'].unique())
print(f'Destab: {len(destab):,} | Neutral: {len(neutral):,}')

In [ ]:
# ── CONTACT MAP HELPERS + SEQUENCE LOADING ────────────────────────────────────
import re as _re

gene_to_acc = {f.name.split('.')[1]: f.name.split('.')[0]
               for f in Path(DDG_DIR).glob('*.ddg_matrix.tsv')}

print('Loading reference sequences...')
seqs = {}
cur_acc = None
with open(REF_FASTA) as fh:
    for line in fh:
        line = line.rstrip()
        if line.startswith('>'):
            parts = line.split('|')
            cur_acc  = parts[1] if len(parts) >= 3 else line[1:].split()[0]
            m_gene   = _re.search(r'GN=(\S+)', line)
            cur_gene = m_gene.group(1) if m_gene else None
            seqs[cur_acc] = []
            if cur_gene and cur_gene not in gene_to_acc:
                gene_to_acc[cur_gene] = cur_acc
        elif cur_acc:
            seqs[cur_acc].append(line)
seqs = {acc: ''.join(s) for acc, s in seqs.items()}
print(f'  {len(seqs):,} sequences | {len(gene_to_acc):,} gene->acc mappings')

# ── Tryptic digestion ─────────────────────────────────────────────────────────
def _tryptic_cuts(seq):
    cuts = [-1]
    for i, aa in enumerate(seq):
        if aa in 'KR' and (i + 1 >= len(seq) or seq[i + 1] != 'P'):
            cuts.append(i)
    cuts.append(len(seq) - 1)
    return cuts

def get_wt_peptides(acc, contact_pos_1based):
    """Return WT tryptic peptide sequences (0- and 1-missed cleavage) covering contact_pos."""
    seq = seqs.get(acc, '')
    if not seq or contact_pos_1based < 1 or contact_pos_1based > len(seq):
        return []
    cuts = _tryptic_cuts(seq)
    idx = contact_pos_1based - 1
    results = set()
    for i in range(len(cuts) - 1):
        for j in range(i + 1, min(i + 3, len(cuts))):
            start = cuts[i] + 1
            end   = cuts[j] + 1
            if start <= idx < end:
                pep = seq[start:end]
                if len(pep) >= 6:
                    results.add(pep)
    return list(results)

# ── Contact map helpers ───────────────────────────────────────────────────────
cm_cache     = {}
nearby_cache = {}

def load_contact_map(acc):
    cdir = Path(CONTACT_DIR)
    for npy_path in sorted(cdir.glob(f'AF-{acc}-*F1.npy')):
        csv_path = npy_path.with_suffix('.csv')
        if not csv_path.exists(): continue
        try:
            meta = pd.read_csv(csv_path, index_col=0)
            dm   = np.load(npy_path)
            p2i  = {int(row['id']): i for i, row in meta.iterrows() if pd.notna(row['id'])}
            return p2i, dm
        except Exception:
            continue
    return None, None

def get_nearby(acc, pos):
    key = (acc, pos)
    if key not in nearby_cache:
        if acc not in cm_cache:
            cm_cache[acc] = load_contact_map(acc)
        p2i, dm = cm_cache[acc]
        if dm is None:
            nearby_cache[key] = []
        else:
            idx = p2i.get(pos)
            if idx is None:
                nearby_cache[key] = []
            else:
                pos_arr = np.array(list(p2i.keys()),   dtype=np.int32)
                idx_arr = np.array(list(p2i.values()), dtype=np.int32)
                d_vals  = dm[idx][idx_arr]
                mask    = (d_vals > 0) & (d_vals < DIST_THRESHOLD) & (pos_arr != pos)
                nearby_cache[key] = pos_arr[mask].tolist()
    return nearby_cache[key]

def find_ri_cols(df):
    found = {}
    for ch in CHANNEL_ORDER + ['126C','132N','132C','133N','134N']:
        if ch in df.columns:
            found[ch] = ch; continue
        cands = [c for c in df.columns if c.startswith('Intensity') and c.endswith(f'_{ch}')]
        if cands: found[ch] = cands[0]
    return found

def get_channel_uuid_map(plex_id):
    pt = tmt[tmt['run_metadata_id'] == plex_id][['tmt_channel','case_submitter_id','sample_type']].drop_duplicates()
    pt = pt[~pt['case_submitter_id'].str.lower().isin(['ref','reference','pooled','pool','nan'])]
    pt['channel'] = pt['tmt_channel'].map(TMT_CHANNEL_MAP)
    pt = pt.dropna(subset=['channel'])
    merged = pt.merge(gdc[['gdc_file_id','case_submitter_id','sample_type']],
                      on=['case_submitter_id','sample_type'], how='left')
    return merged.dropna(subset=['gdc_file_id']).drop_duplicates('channel').set_index('channel')['gdc_file_id'].to_dict()

# Build canonical tryptic peptide set for proteome-level false-positive check
print('Building canonical peptide set...')
canonical_peptides = set()
for _seq in seqs.values():
    _cuts = [-1]
    for _i, _aa in enumerate(_seq):
        if _aa in 'KR' and (_i + 1 >= len(_seq) or _seq[_i + 1] != 'P'):
            _cuts.append(_i)
    _cuts.append(len(_seq) - 1)
    for _i in range(len(_cuts) - 1):
        _pep = _seq[_cuts[_i] + 1 : _cuts[_i + 1] + 1]
        if len(_pep) >= 6:
            canonical_peptides.add(_pep)
print(f'  {len(canonical_peptides):,} canonical tryptic peptides indexed')

print('Helpers defined')


In [ ]:
# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
records = []
n_done = n_no_psm = n_no_wt = 0

for pi, plex_id in enumerate(plex_ids):
    if pi % 5 == 0:
        print(f'  {pi}/{len(plex_ids)} plexes, {len(records):,} records', flush=True)

    psm_files = sorted(glob.glob(os.path.join(RESULTS_BASE, plex_id, '*_1/psm.tsv')))
    if not psm_files:
        psm_files = sorted(glob.glob(os.path.join(RESULTS_BASE, plex_id, 'psm.tsv')))
    if not psm_files:
        n_no_psm += 1; continue

    psm = pd.read_csv(psm_files[0], sep='\t', low_memory=False)
    ri_col_map = find_ri_cols(psm)
    if not ri_col_map: continue

    ch_uuid = get_channel_uuid_map(plex_id)
    channels_with_genomics = {ch for ch, uuid in ch_uuid.items()
                              if uuid in all_processed_uuids}
    if len(channels_with_genomics) < 2: continue

    # Aggregate RI per peptide across all PSMs in the plex
    ri_cols_list = list(ri_col_map.values())
    pep_ri = (psm.groupby('Peptide')[ri_cols_list]
                 .sum()
                 .rename(columns={v: k for k, v in ri_col_map.items()}))

    # Get plex patient missense
    plex_uuids = set(ch_uuid.values())
    plex_miss_frames = [all_miss_indexed.get_group(uid)
                        for uid in plex_uuids if uid in all_miss_indexed.groups]
    plex_miss = pd.concat(plex_miss_frames) if plex_miss_frames else pd.DataFrame()
    if len(plex_miss) == 0: continue

    contact_mask = psm['Entry Name'].astype(str).str.endswith('-contact')
    contact_psm  = psm[contact_mask].copy()
    if len(contact_psm) == 0: continue

    seen = set()
    for _, row in contact_psm.iterrows():
        entry_name = str(row.get('Entry Name', ''))
        protein_id  = str(row.get('Protein ID', ''))
        pep_seq     = str(row.get('Peptide', ''))

        dedup_key = (entry_name, protein_id, pep_seq)
        if dedup_key in seen: continue
        seen.add(dedup_key)

        m_gene = _re.match(r'^([A-Z0-9]+)-contact$', entry_name)
        if not m_gene: continue
        gene = m_gene.group(1)

        m_swap = _re.search(r'-([A-Z])(\d+)([A-Z])-[A-Z0-9]{4}$', protein_id)
        if not m_swap: continue
        wt_aa, contact_pos, alt_aa = m_swap.group(1), int(m_swap.group(2)), m_swap.group(3)

        acc = gene_to_acc.get(gene)
        if not acc: continue

        wt_peps = get_wt_peptides(acc, contact_pos)
        wt_pep  = next((p for p in wt_peps if p in pep_ri.index), None)
        if wt_pep is None:
            n_no_wt += 1; continue
        if pep_seq not in pep_ri.index: continue

        gene_miss = plex_miss[plex_miss['SYMBOL'] == gene].dropna(subset=['pos'])
        if len(gene_miss) == 0: continue

        # ── Key fix: build separate carrier sets per source group ──────────────
        # A swap position may be near BOTH destab and neutral missense in the plex.
        # Mixing them into one carrier_uuids set contaminates source_group labels.
        group_carriers = defaultdict(set)   # source_group -> set of carrier UUIDs
        for miss_pos in gene_miss['pos'].unique():
            nearby = get_nearby(acc, int(miss_pos))
            if contact_pos not in nearby: continue
            matched = gene_miss[gene_miss['pos'] == miss_pos]
            for grp in matched['group'].dropna().unique():
                group_carriers[grp] |= set(matched.loc[matched['group'] == grp, 'sample_id'])

        if not group_carriers: continue

        # Non-carriers = patients with NO nearby missense of any group
        all_carrier_uuids = set().union(*group_carriers.values())
        noncarrier_chs = [ch for ch in channels_with_genomics
                          if ch_uuid.get(ch) not in all_carrier_uuids and ch in ri_col_map]
        if len(noncarrier_chs) < 2: continue

        swap_row = pep_ri.loc[pep_seq]
        wt_row   = pep_ri.loc[wt_pep]

        # Emit one set of records per source group independently
        for source_group, carrier_uuids in group_carriers.items():
            carrier_chs = [ch for ch in channels_with_genomics
                           if ch_uuid.get(ch) in carrier_uuids and ch in ri_col_map]
            if not carrier_chs: continue

            for ch in carrier_chs + noncarrier_chs:
                swap_ri = swap_row.get(ch, 0)
                wt_ri   = wt_row.get(ch, 0)
                if wt_ri <= 0: continue
                raas = swap_ri / wt_ri
                log2_raas = float(np.log2(raas)) if raas > 0 else np.nan
                records.append({
                    'plex_id':      plex_id,
                    'gene':         gene,
                    'contact_pos':  contact_pos,
                    'swap':         f'{wt_aa}{contact_pos}{alt_aa}',
                    'wt_peptide':   wt_pep,
                    'channel':      ch,
                    'source_group': source_group,
                    'is_carrier':   ch in carrier_chs,
                    'swap_ri':      float(swap_ri),
                    'wt_ri':        float(wt_ri),
                    'log2_raas':    log2_raas,
                })
    n_done += 1

df = pd.DataFrame(records)
print(f'Done: {n_done} plexes | {len(df):,} channel records | {n_no_wt} swap/no-WT skips')
if len(df):
    print(df['source_group'].value_counts())
    print(f"Carrier rows: {df['is_carrier'].sum():,} | Non-carrier: {(~df['is_carrier']).sum():,}")


In [ ]:
# ── FLAG SUSPICIOUS SWAPS ─────────────────────────────────────────────────────
# Swaps whose mass difference is within 0.05 Da of a common PTM mass may be
# modification artifacts rather than true amino acid substitutions.
# Use CLEAN_ONLY = True to restrict downstream analysis to unambiguous swaps.

_AA_MASS = {
    'A': 71.03711, 'C': 103.00919, 'D': 115.02694, 'E': 129.04259,
    'F': 147.06841, 'G':  57.02146, 'H': 137.05891, 'I': 113.08406,
    'K': 128.09496, 'L': 113.08406, 'M': 131.04049, 'N': 114.04293,
    'P':  97.05276, 'Q': 128.05858, 'R': 156.10111, 'S':  87.03203,
    'T': 101.04768, 'V':  99.06841, 'W': 186.07931, 'Y': 163.06333,
}
_PTM_MASSES = {
    'methylation':              14.01565,
    'oxidation':                15.99491,
    'dehydrogenation':           2.01565,
    'deamidation':               0.98402,
    'acetylation':              42.01057,
    'phosphorylation':          79.96633,
    'water_addition':           18.01056,
    'methylation+deamidation':  14.99967,
    'double_methylation':       28.03130,
    'oxidation+deamidation':    16.97893,
    'double_oxidation':         31.98983,
}
_PTM_TOL = 0.05  # Da

def _swap_suspicion(swap_str):
    """Return PTM name if swap mass difference is within tolerance, else None."""
    m = __import__('re').match(r'^([A-Z])(\d+)([A-Z])$', swap_str)
    if not m: return None
    wt, alt = m.group(1), m.group(3)
    if wt not in _AA_MASS or alt not in _AA_MASS: return None
    delta = abs(_AA_MASS[alt] - _AA_MASS[wt])
    for ptm, mass in _PTM_MASSES.items():
        if abs(delta - mass) < _PTM_TOL:
            return ptm
    return None

df['suspicion'] = df['swap'].apply(_swap_suspicion)
df['suspicious'] = df['suspicion'].notna()

print('Swap suspicion breakdown:')
print(df.drop_duplicates(['gene','contact_pos','swap'])[['swap','suspicion']]
        .groupby('suspicion').size().sort_values(ascending=False).to_string())
print(f"\nClean swaps: {(~df['suspicious']).sum():,} records")
print(f"Suspicious:  {df['suspicious'].sum():,} records")
print()
print('Suspicious swap types detected:')
print(df[df['suspicious']].drop_duplicates(['gene','contact_pos','swap'])
        .groupby('suspicion')['swap'].count().sort_values(ascending=False).to_string())



In [ ]:
# ── AGGREGATE TO ONE DELTA PER EVENT ──────────────────────────────────────────
# For each (plex, gene, swap): Δ = mean_carrier_log2(swap/WT) - mean_noncarrier_log2(swap/WT)
# This controls for any systematic swap/WT ratio differences unrelated to carrier status.

valid = df[df['log2_raas'].notna()]

def _delta(g):
    c  = g.loc[ g['is_carrier'], 'log2_raas']
    nc = g.loc[~g['is_carrier'], 'log2_raas']
    if len(c) == 0 or len(nc) == 0:
        return pd.Series({'delta': float('nan'), 'n_carrier': len(c), 'n_noncarrier': len(nc)})
    return pd.Series({'delta': c.mean() - nc.mean(),
                      'n_carrier': len(c), 'n_noncarrier': len(nc)})

event_df = (valid
    .groupby(['plex_id','gene','contact_pos','swap','source_group'])
    .apply(_delta)
    .reset_index()
    .dropna(subset=['delta'])
)

print(f'Events with delta: {len(event_df):,}')
print(event_df['source_group'].value_counts().to_string())
print()

DESTAB_GROUPS  = ['Both AM+SPURS', 'SPURS only', 'AM only']
NEUTRAL_GROUPS = ['Neutral']

print('── Δlog2(swap/WT):  carrier − non-carrier ──────────────────────────────')
for grp in DESTAB_GROUPS + NEUTRAL_GROUPS:
    d = event_df[event_df['source_group'] == grp]['delta']
    if len(d) < 3: continue
    t, p = stats.ttest_1samp(d, popmean=0)
    print(f'{grp:<20}  n={len(d):,}  mean={d.mean():.4f}  median={d.median():.4f}  p={p:.2e}')

destab_d  = event_df[event_df['source_group'].isin(DESTAB_GROUPS)]['delta']
neutral_d = event_df[event_df['source_group'].isin(NEUTRAL_GROUPS)]['delta']
if len(destab_d) >= 3 and len(neutral_d) >= 3:
    u, p_mw = stats.mannwhitneyu(destab_d, neutral_d, alternative='greater')
    print(f'\nDestab vs Neutral (MW, greater): p={p_mw:.2e}  '
          f'(n={len(destab_d):,} vs {len(neutral_d):,})')


In [ ]:
# ── PLOTS: violin + median bar, all swaps vs clean only ───────────────────────
import math

DESTAB_GROUPS  = ['Both AM+SPURS', 'SPURS only', 'AM only']
NEUTRAL_GROUPS = ['Neutral']
GROUP_SPECS = [
    ('Both AM+SPURS', '#d62728'),
    ('SPURS only',    '#ff7f0e'),
    ('AM only',       '#9467bd'),
    ('Neutral',       '#2ca02c'),
]

def make_event_df(use_clean):
    _df = df[~df['suspicious']] if use_clean else df
    valid = _df[_df['log2_raas'].notna()]
    def _delta(g):
        c  = g.loc[ g['is_carrier'], 'log2_raas']
        nc = g.loc[~g['is_carrier'], 'log2_raas']
        if len(c) == 0 or len(nc) == 0:
            return pd.Series({'delta': float('nan'), 'n_carrier': len(c), 'n_noncarrier': len(nc)})
        return pd.Series({'delta': c.mean() - nc.mean(),
                          'n_carrier': len(c), 'n_noncarrier': len(nc)})
    return (valid
        .groupby(['plex_id','gene','contact_pos','swap','source_group'])
        .apply(_delta)
        .reset_index()
        .dropna(subset=['delta']))

edf_all   = make_event_df(use_clean=False)
edf_clean = make_event_df(use_clean=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for row_i, (edf, label) in enumerate([(edf_all, 'All swaps'), (edf_clean, 'Clean swaps only')]):
    ax_vio = axes[row_i, 0]
    ax_bar = axes[row_i, 1]

    # Violin
    violin_data, violin_colors, xtick_labels = [], [], []
    for grp, color in GROUP_SPECS:
        d = edf[edf['source_group'] == grp]['delta']
        if len(d) < 3: continue
        violin_data.append(d.values)
        violin_colors.append(color)
        xtick_labels.append(f'{grp}\nn={len(d):,}')
    if violin_data:
        parts = ax_vio.violinplot(violin_data, positions=range(len(violin_data)),
                                  showmedians=True, showextrema=False)
        for pc, col in zip(parts['bodies'], violin_colors):
            pc.set_facecolor(col); pc.set_alpha(0.7)
        parts['cmedians'].set_color('black'); parts['cmedians'].set_linewidth(2)
        for i, d in enumerate(violin_data):
            ax_vio.scatter([i]*len(d), d, color='k', alpha=0.25, s=6, zorder=3)
    ax_vio.axhline(0, color='k', lw=0.8, ls='--')
    ax_vio.set_xticks(range(len(xtick_labels)))
    ax_vio.set_xticklabels(xtick_labels, fontsize=8)
    ax_vio.set_ylabel('Δ log2(swap/WT): carrier − non-carrier')
    ax_vio.set_title(f'{label} — distribution (median line)')

    # Median bar with IQR error bars (consistent with violin median line)
    bar_i = 0
    for grp, color in GROUP_SPECS:
        d = edf[edf['source_group'] == grp]['delta'].dropna()
        if len(d) == 0: continue
        med = d.median()
        q1, q3 = d.quantile(0.25), d.quantile(0.75)
        t, p = stats.ttest_1samp(d, popmean=0) if len(d) >= 3 else (float('nan'), float('nan'))
        ax_bar.bar(bar_i, med, color=color, alpha=0.85)
        ax_bar.errorbar(bar_i, med, yerr=[[med - q1], [q3 - med]],
                        fmt='none', color='black', capsize=5, linewidth=1.5)
        pstr = f'p={p:.3f}' if not math.isnan(p) else ''
        ax_bar.text(bar_i, q3 + 0.02,
                    f'n={len(d):,}\n{pstr}', ha='center', va='bottom', fontsize=7.5)
        bar_i += 1
    ax_bar.axhline(0, color='k', lw=0.8, ls='--')
    ax_bar.set_xticks(range(bar_i))
    ax_bar.set_xticklabels([g[0] for g in GROUP_SPECS[:bar_i]], fontsize=8, rotation=15, ha='right')
    ax_bar.set_ylabel('median Δ log2(swap/WT)  [IQR error bars]')
    ax_bar.set_title(f'{label} — median + IQR (t-test vs 0)')

plt.suptitle('Contact-predicted SAAP: carrier vs non-carrier RAAS delta\n'
             'Δ = mean_carrier[log2(swap/WT)] − mean_noncarrier[log2(swap/WT)]',
             fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'contact_saap_raas_delta.pdf'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
import re as _re2
issues = []
n_checked = 0

for (gene, contact_pos, swap), grp in df.groupby(['gene','contact_pos','swap']):
    row0 = grp.iloc[0]
    wt_pep  = row0['wt_peptide']
    pep_seq = None  # swap peptide seq not stored — checked via swap string

    # Parse wt_aa and alt_aa from swap string e.g. "K245A"
    m = _re2.match(r'^([A-Z])(\d+)([A-Z])$', swap)
    if not m:
        issues.append(f'Unparseable swap string: {swap} in {gene}')
        continue
    wt_aa, pos, alt_aa = m.group(1), int(m.group(2)), m.group(3)

    acc = gene_to_acc.get(gene)
    seq = seqs.get(acc, '') if acc else ''

    # 1. contact_pos within protein length
    if seq and (contact_pos < 1 or contact_pos > len(seq)):
        issues.append(f'{gene} {swap}: contact_pos {contact_pos} out of range (len={len(seq)})')
        continue

    # 2. wt_aa matches reference sequence at contact_pos
    if seq and seq[contact_pos - 1] != wt_aa:
        issues.append(f'{gene} {swap}: wt_aa={wt_aa} but ref has {seq[contact_pos-1]} at pos {contact_pos}')

    # 3. wt_peptide should contain wt_aa somewhere (and NOT alt_aa at that position)
    if seq and wt_pep:
        # Find where wt_pep maps in the protein to get the contact position within peptide
        pep_start = seq.find(wt_pep)
        if pep_start == -1:
            issues.append(f'{gene} {swap}: wt_peptide not found in reference sequence')
        else:
            pep_idx = contact_pos - 1 - pep_start
            if 0 <= pep_idx < len(wt_pep):
                if wt_pep[pep_idx] != wt_aa:
                    issues.append(f'{gene} {swap}: wt_peptide has {wt_pep[pep_idx]} at contact pos, expected {wt_aa}')
                if wt_pep[pep_idx] == alt_aa:
                    issues.append(f'{gene} {swap}: wt_peptide already contains alt_aa {alt_aa} at contact pos!')

    # 4. Self-swap
    if wt_aa == alt_aa:
        issues.append(f'{gene} {swap}: wt_aa == alt_aa ({wt_aa})')

    # 5. Excluded swap slipped through
    excluded = {('N','D'),('D','N'),('Q','E'),('E','Q'),('I','L'),('L','I')}
    if (wt_aa, alt_aa) in excluded:
        issues.append(f'{gene} {swap}: excluded swap ({wt_aa}->{alt_aa}) found in results')
    if wt_aa in 'KR' or alt_aa in 'KR':
        issues.append(f'{gene} {swap}: K/R swap found in results')

    n_checked += 1

print(f'Checked {n_checked} unique (gene, contact_pos, swap) combinations')
if issues:
    print(f'\n{len(issues)} issues found:')
    for iss in issues[:50]:
        print(f'  {iss}')
    if len(issues) > 50:
        print(f'  ... and {len(issues)-50} more')
    # 7. Swap peptide sequence matches a canonical human tryptic peptide
    #    (check via wt_peptide with alt_aa substituted back in)
    if seq and wt_pep:
        pep_start = seq.find(wt_pep)
        if pep_start >= 0:
            pep_idx = contact_pos - 1 - pep_start
            if 0 <= pep_idx < len(wt_pep):
                swap_pep = wt_pep[:pep_idx] + alt_aa + wt_pep[pep_idx+1:]
                if swap_pep in canonical_peptides:
                    issues.append(f'{gene} {swap}: swap peptide {swap_pep} exists as canonical human peptide!')

else:
    print('No issues found.')


In [ ]:
# ── SWAP HEATMAP: from AA (x) → to AA (y), color = unique swap peptides ───────
import re as _re3

AA_ORDER = list('ACDEFGHIKLMNPQRSTVWY')

# Parse wt_aa and alt_aa from swap column, count unique (gene, contact_pos, swap)
swap_counts = (df.drop_duplicates(['gene','contact_pos','swap'])
                 ['swap']
                 .str.extract(r'^([A-Z])\d+([A-Z])$')
                 .rename(columns={0:'wt', 1:'alt'})
                 .dropna()
                 .groupby(['wt','alt'])
                 .size()
                 .reset_index(name='n'))

# Build 20x20 matrix
mat = pd.DataFrame(0, index=AA_ORDER, columns=AA_ORDER)
for _, row in swap_counts.iterrows():
    if row['wt'] in mat.index and row['alt'] in mat.columns:
        mat.loc[row['wt'], row['alt']] = row['n']

import numpy as np
mat_np = mat.values.astype(float)
mat_np[mat_np == 0] = np.nan  # grey for undetected

fig, ax = plt.subplots(figsize=(10, 9))
cmap = plt.cm.YlOrRd.copy()
cmap.set_bad('lightgrey')

im = ax.imshow(mat_np, cmap=cmap, aspect='equal')
plt.colorbar(im, ax=ax, label='# unique swap peptides detected')

ax.set_xticks(range(len(AA_ORDER)))
ax.set_yticks(range(len(AA_ORDER)))
ax.set_xticklabels(AA_ORDER, fontsize=9)
ax.set_yticklabels(AA_ORDER, fontsize=9)
ax.set_xlabel('From (WT amino acid)', fontsize=11)
ax.set_ylabel('To (swap amino acid)', fontsize=11)
ax.set_title('Contact-predicted SAAP detections\nby amino acid swap type', fontsize=12)

# Annotate cells with count where > 0
for i, wt in enumerate(AA_ORDER):
    for j, alt in enumerate(AA_ORDER):
        v = mat.loc[wt, alt]
        if v > 0:
            ax.text(j, i, str(v), ha='center', va='center',
                    fontsize=6, color='black' if v < mat_np[~np.isnan(mat_np)].max()*0.6 else 'white')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'contact_saap_swap_heatmap.pdf'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Total unique swap peptides: {df.drop_duplicates(['gene','contact_pos','swap']).shape[0]:,}")
print("\nTop swap types:")
print(swap_counts.sort_values('n', ascending=False).head(10).to_string(index=False))


In [ ]:
# ── PSM CONFIDENCE SCORES + LOCALIZATION PROXY ────────────────────────────────
# Re-reads psm.tsv for contact PSMs to get:
#   PEP (1 - PeptideProphet prob), hyperscore, matched/total ion fraction
# Localization proxy: is the swap position flanked by >=2 theoretical ions
#   on each side? (i.e. >= 2 b-ions N-terminal and >= 2 y-ions C-terminal)
# This is a necessary (not sufficient) condition for unambiguous localization.

score_records = []
for plex_id in plex_ids:
    psm_files = sorted(glob.glob(os.path.join(RESULTS_BASE, plex_id, '*_1/psm.tsv')))
    if not psm_files: continue
    try:
        psm = pd.read_csv(psm_files[0], sep='\t', low_memory=False)
    except Exception: continue

    if not score_records:  # print column names once from first plex
        score_cols = [c for c in psm.columns if any(k in c.lower()
                      for k in ['prob','hyper','match','ion','expect','score','pep'])]
        print(f"PSM score columns available: {score_cols}")
    contact_mask = psm['Entry Name'].astype(str).str.endswith('-contact')
    contact_psm  = psm[contact_mask].copy()
    if len(contact_psm) == 0: continue

    for _, row in contact_psm.iterrows():
        entry_name = str(row.get('Entry Name', ''))
        protein_id = str(row.get('Protein ID', ''))
        pep_seq    = str(row.get('Peptide', ''))

        m_gene = _re.match(r'^([A-Z0-9]+)-contact$', entry_name)
        m_swap = _re.search(r'-([A-Z])(\d+)([A-Z])-[A-Z0-9]{4}$', protein_id)
        if not m_gene or not m_swap: continue

        gene = m_gene.group(1)
        wt_aa, contact_pos, alt_aa = m_swap.group(1), int(m_swap.group(2)), m_swap.group(3)

        # PSM confidence scores — try multiple FragPipe column name variants
        def _get(*keys):
            for k in keys:
                v = row.get(k, np.nan)
                if pd.notna(v): return pd.to_numeric(v, errors='coerce')
            return np.nan
        prob       = _get('PeptideProphet Probability', 'Probability')
        pep        = float(1 - prob) if pd.notna(prob) else np.nan
        hyperscore = _get('Hyperscore', 'MSFragger:Hyperscore', 'hyperscore')
        n_matched  = _get('Number of Matched Ions', 'Matched Ions', 'Matched Fragment Ions')
        n_total    = _get('Total Number of Ions', 'Total Ions', 'Total Fragment Ions')
        matched_frac = float(n_matched / n_total) if (pd.notna(n_matched) and pd.notna(n_total) and n_total > 0) else np.nan
        expectation  = _get('Expectation', 'MSFragger:Expectation', 'e-value')

        # Localization proxy: find swap position within peptide
        acc = gene_to_acc.get(gene)
        seq = seqs.get(acc, '') if acc else ''
        wt_peps = get_wt_peptides(acc, contact_pos) if acc else []
        wt_pep  = next((p for p in wt_peps if seq.find(p) >= 0), None)

        swap_pos_in_pep = np.nan
        localizable = False
        if wt_pep and seq:
            pep_start = seq.find(wt_pep)
            if pep_start >= 0:
                swap_pos_in_pep = (contact_pos - 1) - pep_start  # 0-indexed within peptide
                n_left  = int(swap_pos_in_pep)                   # b-ions not containing swap
                n_right = len(wt_pep) - int(swap_pos_in_pep) - 1 # y-ions not containing swap
                localizable = min(n_left, n_right) >= 2

        score_records.append({
            'plex_id':         plex_id,
            'gene':            gene,
            'contact_pos':     contact_pos,
            'swap':            f'{wt_aa}{contact_pos}{alt_aa}',
            'wt_peptide':      wt_pep or '',
            'swap_peptide':    pep_seq,
            'pep':             pep,
            'hyperscore':      float(hyperscore) if pd.notna(hyperscore) else np.nan,
            'matched_frac':    matched_frac,
            'expectation':     float(expectation) if pd.notna(expectation) else np.nan,
            'swap_pos_in_pep': swap_pos_in_pep,
            'localizable':     localizable,
        })

scores_df = pd.DataFrame(score_records)
print(f"Contact PSMs: {len(scores_df):,} across {scores_df['plex_id'].nunique()} plexes")
print(f"Localizable (>=2 ions each side of swap): {scores_df['localizable'].sum():,} / {len(scores_df):,} "
      f"({100*scores_df['localizable'].mean():.1f}%)")
print(f"Median matched ion fraction: {scores_df['matched_frac'].median():.3f}")
print(f"Median PEP: {scores_df['pep'].median():.4f}")
print(f"\nBest-confidence PSMs (PEP<0.01, localizable, matched_frac>0.3):")
hi = scores_df[scores_df['localizable'] & (scores_df['pep'] < 0.01) & (scores_df['matched_frac'] > 0.3)]
print(hi[['gene','swap','pep','hyperscore','matched_frac']].drop_duplicates(['gene','swap'])
        .sort_values('pep').head(20).to_string(index=False))


In [ ]:
# ── FINAL SUMMARY TABLE ────────────────────────────────────────────────────────
# One row per (gene, swap, source_group) aggregated across all plexes.
# Columns: peptides, carrier/noncarrier channels, missense scores,
#          plex counts, PSM confidence, localization proxy.

# 1. RAAS delta per event (from event_df computed in stats cell)
event_stats = (event_df
    .groupby(['gene','contact_pos','swap','source_group'])
    .agg(
        n_events        = ('delta', 'count'),
        median_delta    = ('delta', 'median'),
        mean_delta      = ('delta', 'mean'),
        n_carrier_ch    = ('n_carrier',    'sum'),
        n_noncarrier_ch = ('n_noncarrier', 'sum'),
    )
    .reset_index())

# 2. PSM confidence scores per (gene, swap) — best across all PSMs
score_stats = (scores_df
    .groupby(['gene','contact_pos','swap'])
    .agg(
        swap_peptide    = ('swap_peptide', 'first'),
        wt_peptide      = ('wt_peptide',   'first'),
        best_pep        = ('pep',          'min'),
        best_hyperscore = ('hyperscore',   'max'),
        median_matched_frac = ('matched_frac', 'median'),
        n_psms          = ('pep',          'count'),
        n_plexes_detected = ('plex_id',    'nunique'),
        pct_localizable = ('localizable',  'mean'),
    )
    .reset_index())

# 3. Missense scores — for each (gene, contact_pos, source_group)
#    look up the nearby missense in the full missense table
miss_stats_records = []
for (gene, cpos, grp), grp_df in df.groupby(['gene','contact_pos','source_group']):
    acc = gene_to_acc.get(gene)
    if not acc: continue
    gene_miss = all_miss[
        (all_miss['SYMBOL'] == gene) &
        (all_miss['group']  == grp)
    ].dropna(subset=['pos'])
    nearby_miss = gene_miss[[cpos in get_nearby(acc, int(p))
                              for p in gene_miss['pos']]]
    if len(nearby_miss) == 0: continue
    miss_stats_records.append({
        'gene':            gene,
        'contact_pos':     cpos,
        'source_group':    grp,
        'missense_am':     nearby_miss['am_pathogenicity'].median(),
        'missense_spurs':  nearby_miss['spurs_ddg'].median(),
        'missense_vaf':    nearby_miss['VAF'].median(),
        'missense_gnomad': nearby_miss['gnomADe_AF'].median(),
        'n_patients_with_missense': nearby_miss['sample_id'].nunique(),
    })
miss_stats = pd.DataFrame(miss_stats_records)

# 4. Number of plexes where the missense exists (regardless of detection)
plex_miss_counts = (all_miss[all_miss['SYMBOL'].isin(df['gene'].unique())]
    .groupby('SYMBOL')['sample_id']
    .nunique()
    .rename('n_patients_total')
    .reset_index()
    .rename(columns={'SYMBOL':'gene'}))

# 5. Suspicious flag per swap
sus_map = (df[['swap','suspicious','suspicion']]
           .drop_duplicates('swap')
           .set_index('swap'))

# Merge everything
summary = (event_stats
    .merge(score_stats,  on=['gene','contact_pos','swap'], how='left')
    .merge(miss_stats,   on=['gene','contact_pos','source_group'], how='left')
    .merge(plex_miss_counts, on='gene', how='left'))
summary['suspicious'] = summary['swap'].map(sus_map['suspicious']).fillna(False)
summary['suspicion']  = summary['swap'].map(sus_map['suspicion'])

# Sort by best_pep then median_delta descending
summary = summary.sort_values(['best_pep','median_delta'], ascending=[True, False])

cols = ['gene','swap','swap_peptide','wt_peptide','source_group',
        'n_events','n_carrier_ch','n_noncarrier_ch',
        'median_delta','mean_delta',
        'missense_am','missense_spurs','missense_vaf','missense_gnomad',
        'n_patients_with_missense','n_patients_total',
        'n_plexes_detected','n_psms',
        'best_pep','best_hyperscore','median_matched_frac','pct_localizable',
        'suspicious','suspicion']
summary = summary[[c for c in cols if c in summary.columns]]

# Filter Ig genes — somatic hypermutation makes swaps uninterpretable
IG_PREFIXES = ('IGH','IGL','IGK','IGHA','IGHG','IGHM','IGLC','IGKC')
ig_mask = summary['gene'].str.startswith(IG_PREFIXES)
summary_clean = summary[~ig_mask].copy()
print(f"Summary table: {len(summary):,} rows total, {ig_mask.sum()} Ig rows removed")
print(f"Clean summary: {len(summary_clean):,} rows")

print("\nTop 20 clean hits (localizable, non-suspicious, non-Ig):")
top = summary_clean[~summary_clean['suspicious'] & (summary_clean['pct_localizable'] > 0.5)]
print(top.head(20)[['gene','swap','source_group','n_events','median_delta',
                      'best_pep','pct_localizable','missense_am','missense_spurs']].to_string(index=False))

summary.to_csv(os.path.join(OUT_DIR, 'contact_saap_summary.tsv'), sep='\t', index=False)
summary_clean.to_csv(os.path.join(OUT_DIR, 'contact_saap_summary_clean.tsv'), sep='\t', index=False)
print(f"\nSaved to {OUT_DIR}/contact_saap_summary*.tsv")


In [ ]:
# ── SAVE ──────────────────────────────────────────────────────────────────────
df.to_csv(os.path.join(OUT_DIR, 'contact_saap_records.tsv'), sep='\t', index=False)
print(f'Saved to {OUT_DIR}')